# AIR Homework 1 (Task 4)

## Setup

This notebook covers Task 4. Run the cells below before you start. They set up the notebook, load our working environment from `scene.py`, import your code, and recreate what we need from Tasks 1-3, including the motion plan from the start configuration to the home configuration.

**NOTE:** The setup uses your code in `task1.py`, `task2.py`, and `task3.py`, so complete Tasks 1-3 before you start this task.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

In [ ]:
# load our working environment from scene.py (see start_here.ipynb)
from scene import HOME_CONFIG, data, model

import matplotlib.pyplot as plt
import mujoco
import numpy as np
from tqdm.auto import tqdm

In [ ]:
# import your code. the setup below uses your code from Tasks 1-3 (t1, t2, t3), and
# the rest of this notebook uses your code for this task (t4), e.g. t4.WaypointPolicy
import task1 as t1
import task2 as t2
import task3 as t3
import task4 as t4

In [ ]:
# from Task 1
import unittest

# used by the sanity checks
test = unittest.TestCase()


def get_check_data():
    """Get a separate simulation state at the home configuration, so that the sanity
    checks don't affect the rest of this notebook."""
    check_data = mujoco.MjData(model)
    mujoco.mj_resetDataKeyframe(model, check_data, model.key("robot_home").id)
    mujoco.mj_forward(model, check_data)
    return check_data


# the pre-grasp pose above the red box, at the home keyframe
mujoco.mj_resetDataKeyframe(model, data, model.key("robot_home").id)
mujoco.mj_forward(model, data)
above_block_pos, above_block_rot_mat = t1.get_above_block_pose(data)

In [ ]:
# from Task 3

# set an alternative orientation above the box (90 degrees rotated around z axis)
above_block_rot_mat2 = t3.get_above_block_rot_mat2(above_block_rot_mat)

# define pre-grasp poses
pre_grasp_poses = [
    (above_block_pos, above_block_rot_mat),
    (above_block_pos, above_block_rot_mat2),
]

# define grasp poses to be slightly lower than pre-grasp but not touching the box
on_block_pos = t3.get_on_block_pos(above_block_pos)
grasp_poses = [
    (on_block_pos, above_block_rot_mat),
    (on_block_pos, above_block_rot_mat2),
]

# plan a motion from the initial configuration to the home configuration
mujoco.mj_resetData(model, data)  # Reset state and time.
mujoco.mj_forward(model, data)
planner = t3.get_planner_for_problem(
    start_q=[0, 0, 0, 0, 0, 0],
    goal_q=HOME_CONFIG,
    model=model,
    data=data,
    controlled_bodies_prefix=["robot_", "gripper_"],
    check_resolution=0.01,
)
waypoints, status = t3.plan_motion(planner, time_limit=5.0)

## Task 4: Task Execution Policy

It is now time to put everything together into a single policy that executes the full pick-and-place task. Specifically, the robot's task is to:
1. Place the red box into the green bin.
2. Avoid colliding with the obstacles on the table.
3. Return to the home configuration after placing the box and hold for 1 second.

We will implement this as a sequential state machine, where each state corresponds to a step in the manipulation sequence outlined earlier. While this is a common approach to designing complex robotic behavior that has implementations in common robotics frameworks such as [ROS SMACH](https://wiki.ros.org/smach), we will implement our own simplified version here for educational purposes.

Below we have implemented two classes:
1. `PolicySMNode` represents a single node in a state machine which can initialize its policy and can determine when it has completed its task.
2. `PolicyStateMachine` represents a sequential state machine that executes a series of `PolicySMNode` objects in order.

**NOTE:** Throughout this task, we will run intermediate tasks to verify that each component is working correctly. Your motion plans might be longer or shorter depending on the planner and parameters you selected. You may need to change the `duration` parameter in the `run_and_render_policy` function to ensure that the robot has enough time to complete each motion plan.

`PolicySMNode` and `SMPolicy` are provided in `task4.py`.

### Motion Plans as a Sequential State Machine

Let's start off by implementing a state-machine policy to execute a single motion plan given the waypoints. Executing a motion plan is essentially following a series of waypoints using our PID controller. Thus, each node is simply a PID control to the next waypoint. Let's implement that node Now.

**TODO:** implement `GoToConfigSMNode` in `task4.py`.

In [ ]:
# sanity check: GoToConfigSMNode
check_data = get_check_data()  # the robot is at the home configuration
robot_ctrl_idx = t2.get_entity_ctrl_idx(model, "robot")

# the node is complete at its target, but not when the target is far away
test.assertTrue(t4.GoToConfigSMNode(HOME_CONFIG).is_complete(model, check_data))
far_target_node = t4.GoToConfigSMNode(HOME_CONFIG + 0.1)
test.assertFalse(far_target_node.is_complete(model, check_data))

# its policy pushes every joint toward the target
far_target_node.get_policy(model, check_data).set_control(check_data)
test.assertTrue(np.all(check_data.ctrl[robot_ctrl_idx] > 0))

print(f"GoToConfigSMNode sanity checks passed. torques toward the target = {check_data.ctrl[robot_ctrl_idx].round(3)}")

Now, let's create a subclass `SMPolicy` for executing a sequence of waypoints:

**TODO:** implement `WaypointPolicy` in `task4.py`.

Let's test it out by running the motion plan we generated earlier from the start configuration to the home configuration:

In [ ]:
# After the state machine policy has finished executing all its nodes, we
# can use the following callback function to end the simulation. If we
# don't do this, the simulation will continue running until it has fulfilled
# the input duration, which is not necessary once the policy is complete.

# initialize the Mujoco simulation to the home configuration
wp_policy = t4.WaypointPolicy(
    model,
    waypoints,
    waypoint_tolerance=5e-2,
)

# render the execution of the state machine policy
t2.run_and_render_policy(
    model,
    data,
    wp_policy,
    duration=10.0,
    framerate=60,
    camera="frontview",
    callback=t4.end_on_sm_complete_callback,
)

The robot may knock over an obstacle while executing the motion plan. This is highly possible since we have not yet tuned the planner parameters for this specific task. This will need to be addressed in your final policy.

### Motion Planning Nodes

We will now implement a node that plans a motion and executes it using the waypoint policy we just implemented. This node will use all the motion planning components we implemented in the previous task.

Let's start with a node that plans and moves to a target configuration from the current configuration. To implement this node, create a planner, call a plan, replan if necessary, and execute the resulting waypoints using the `SMPolicy` we just implemented.

**NOTE:** This is where we can see why we create the planner separately from planning the motion. If we created the planner inside this node, we would have to re-initialize it every time we wanted to plan a motion. However, OMPL allows us to continue planning from the previous state of the planner, which can significantly improve planning times in some cases.

**TODO:** implement `MotionPlanToConfigSMNode` in `task4.py`.

Let's test out our motion planning node by moving the robot from the start configuration to the home configuration:

**TODO:** implement `get_home_policy` in `task4.py`.

In [ ]:
# single node state machine policy to move to home configuration
mp_policy = t4.get_home_policy(model)

t2.run_and_render_policy(
    model,
    data,
    mp_policy,
    duration=10.0,
    framerate=60,
    camera="frontview",
    callback=t4.end_on_sm_complete_callback
)

Now let's implement the same node, but for a target end-effector pose instead of a configuration. Use the configuration selector function you implemented earlier.

**TODO:** implement `MotionPlanToPoseSMNode` in `task4.py`.

Now let's send the robot to a target end-effector pose above the red box (pre-grasp pose):

**TODO:** implement `get_to_box_policy` in `task4.py`.

In [ ]:
to_box_policy = t4.get_to_box_policy(model, pre_grasp_poses)

t2.run_and_render_policy(
    model,
    data,
    to_box_policy,
    duration=15.0,
    framerate=60,
    camera="frontview",
    callback=t4.end_on_sm_complete_callback
)

### Gripper Nodes

Since we did not include force feedback in our gripper controller, we will assume that the gripper always successfully opens and closes. Therefore, our gripper nodes will simply execute the gripper controller for a fixed duration so that the gripper has enough time to fully open or close.

Let's start by implementing a node that runs some policy for a fixed duration:

**TODO:** implement `DurationConfigSMNode` in `task4.py`.

In [ ]:
# sanity check: DurationConfigSMNode
check_data = get_check_data()
duration_node = t4.DurationConfigSMNode(t3.StayInPlacePolicy(model), duration=1.0)

# the time is counted from when the node gets its policy, which is not always time 0
check_data.time = 5.0
duration_node.get_policy(model, check_data)
test.assertFalse(duration_node.is_complete(model, check_data))

check_data.time = 5.9
test.assertFalse(duration_node.is_complete(model, check_data))

check_data.time = 6.1
test.assertTrue(duration_node.is_complete(model, check_data))

print("DurationConfigSMNode sanity checks passed. a 1 second node started at 5.0 completes after 6.0")

Now let's implement a policy that controls the gripper. Like before, we will need to control the gripper as well as the robot to ensure that the robot remains stable while operating the gripper.

**TODO:** implement `GripperControlPolicy` in `task4.py`.

In [ ]:
# sanity check: GripperControlPolicy
check_data = get_check_data()
robot_ctrl_idx = t2.get_entity_ctrl_idx(model, "robot")
gripper_ctrl_idx = t2.get_entity_ctrl_idx(model, "gripper")

# the gripper closes and opens
t4.GripperControlPolicy(model, close=True).set_control(check_data)
closed_ctrl = check_data.ctrl[gripper_ctrl_idx[0]]
t4.GripperControlPolicy(model, close=False).set_control(check_data)
open_ctrl = check_data.ctrl[gripper_ctrl_idx[0]]
test.assertEqual(closed_ctrl, 255)
test.assertEqual(open_ctrl, 0)

# the robot holds its configuration: if we nudge it, every joint is pushed back
gripper_policy = t4.GripperControlPolicy(model, close=True)
gripper_policy.set_control(check_data)
check_data.qpos[t1.get_entity_qpos_idx(model, "robot")] += 0.1
gripper_policy.set_control(check_data)
test.assertTrue(np.all(check_data.ctrl[robot_ctrl_idx] < 0))

print(f"GripperControlPolicy sanity checks passed. closed = {closed_ctrl:.0f}, open = {open_ctrl:.0f}")

We are now ready to pick up the block!

From the pre-grasp position above the block, we will perform a motion planning node to the grasp position, followed by a gripper close node to grab the block, then another motion planning node to the lift position. Don't forget to specify the attached body after closing the gripper. Also, we can finally set a node at the end of the motion plan to hold the robot at the final configuration.

**NOTE:** The red box should still be in contact with the table after closing the gripper on it. Therefore, it should be allowed to collide with the table during the lifting motion.

**TODO:** implement `get_lift_box_policy` in `task4.py`.

In [ ]:
lift_box_policy = t4.get_lift_box_policy(model, pre_grasp_poses, grasp_poses)

t2.run_and_render_policy(
    model,
    data,
    lift_box_policy,
    duration=15.0,
    framerate=60,
    camera="frontview",
    callback=t4.end_on_sm_complete_callback,
)

### Pick-and-Place Execution

Having picked up the block, we are now ready to place it in the green bin. We already have all the nodes we need. All that's left is to find the relevant poses above the green bin before implementing the final policy.

**TODO:** implement `get_above_bin_pos` in `task4.py`.

In [ ]:
# find the position above the green bin
above_bin_pos = t4.get_above_bin_pos(data)

place_poses = [
    # use the same rotation as the grasp pose
    (above_bin_pos, above_block_rot_mat),
    (above_bin_pos, above_block_rot_mat2),
]

In [ ]:
# sanity check: get_above_bin_pos
bin_pos = get_check_data().body("green_bin").xpos

# the bin is centered at bin_pos. its walls are 8 cm from the center and reach a height of 0.83
test.assertLess(np.abs(above_bin_pos[:2] - bin_pos[:2]).max(), 0.08)
test.assertGreater(above_bin_pos[2], 0.83)

print(f"get_above_bin_pos sanity checks passed. position above the bin = {np.round(above_bin_pos, 3)}")

Implement the final pick-and-place policy as a sequential state machine using all the nodes we have implemented so far. Set your hyperparameters as needed to ensure successful execution.

**TODO:** implement `PickAndPlacePolicy` in `task4.py`.

Let's see our policy in action.

In [ ]:
# initialize and run the pick-and-place policy
pnp_policy = t4.PickAndPlacePolicy(model, pre_grasp_poses, grasp_poses, place_poses)
t2.run_and_render_policy(
    model,
    data,
    pnp_policy,
    duration=60.0,
    framerate=60,
    camera="frontview",
    callback=t4.end_on_sm_complete_callback,
)

Didn't work? Try again! PID control isn't perfect and may be unstable or inaccurate at times, causing error in the motion trajectory that leads to collision. Furthermore, our motion planner might not find an exact solution every time due to the stochastic nature of these algorithms. Feel free to modify your planning parameters and experiment with different configurations to achieve more consistent results. You may also alter the planning procedure or controller as you see fit. Here are some tips on how to do this:
1. **Replanning**: If the motion planner fails to find an exact solution, clear the previous plan and try again.
2. **Concatenating approximate plans**: If the motion planner returns an approximate solution, you can replan from the end of the approximate solution to the goal configuration. This is usually a shorter distance to the goal, making it easier to find a solution.
3. **Padding obstacles**: If the robot is consistently colliding with obstacles, consider adding padding around the obstacles in the collision checker to create a safety buffer (though this may reduce the available space for the robot to navigate).
4. **Change the Planner**: OMPL offers a variety of sampling-based motion planners. Some planners may perform better than others in certain environments. Experiment with different planners to see if one works better for your specific scenario (see available planners [here](https://ompl.kavrakilab.org/planners.html)).
5. **Tuning PID gains**: Perform more comprehensive tuning of the PID gains specifically for the pick-and-place task, as the dynamics may differ from those observed during static position control.

There are more accurate controllers (e.g. [model predictive control](https://en.wikipedia.org/wiki/Model_predictive_control)) as well as parallelized and optimized motion planners to improve tree expansion efficiency (e.g., [cuROBO](https://curobo.org/index.html)) that are beyond the scope of this assignment.

**NOTE**: You may notice some errors indicating the agent was unable to find a collision-free IK solution. This is likely because an obstacle fell on the robot during execution, and now every configuration is a collision configuration. See this as any other failure case.

### Empirical Evaluation

Remember that we are using non-deterministic algorithms. Therefore, execution will differ every time and may lead to different outcomes. We will now evaluate the performance of our pick-and-place policy over multiple trials. We will measure two key metrics:
1. **Success Rate**: The percentage of trials in which the robot successfully picks up the red box and places it in the green bin without any collisions.
2. **Average Execution Time**: The average time taken to complete the pick-and-place task across all successful trials.

Below we implement a runtime callback for evaluating performance throughout execution. Specifically:
- We check for collisions at each timestep. If a collision with an obstacle is detected, we mark the trial as a failure.
- We check that the red box is located within the green bin's walls and in contact with the bottom of the bin at the end of the trial to determine if the placement was successful.
- We check that in the last second of execution, the robot remains stable at the home configuration.
- We terminate execution when the policy has finished executing all nodes in the state machine.

In [ ]:
# provided in task4.py
eval_cb = t4.EvaluatePolicyCallback()

The code below will execute your policy multiple times and measure our evaluation metrics.

In [ ]:
import time
import traceback

NUM_TRIALS = 40

# a container to hold trial results
trial_results = []

# run multiple trials of the pick-and-place policy and collect evaluation data
for _ in tqdm(range(NUM_TRIALS), desc="Evaluating Pick-and-Place Policy"):
    # reset policy and evaluation callback
    pnp_policy.reset()
    eval_cb.reset()

    # record start time
    start_time = time.time()
    try:
        # attempt to run the pick-and-place policy
        t2.run_and_render_policy(
            model,
            data,
            pnp_policy,
            duration=60.0,
            framerate=0,  # change to a low positive framerate for debugging
            camera="frontview",
            callback=eval_cb,
        )
    except Exception as e:
        print("Exception during policy execution:", e)
        traceback.print_exc()

        end_time = float("inf")  # indicate failure with infinite execution time
    else:
        end_time = time.time()  # record end time

    trial_duration = end_time - start_time
    simulation_time = data.time

    trial_results.append(
        {
            "has_collided": eval_cb.has_collided,
            "success": eval_cb.task_success,
            "trial_duration": trial_duration,
            "simulation_time": simulation_time,
        }
    )

Below we visualize success rate.

**NOTE**: you are expected to successfully complete the task without collision about 75% of the time. If you are consistently failing, try tuning the node parameters and consider the tips above.

In [ ]:
num_successful_trials = sum(1 for result in trial_results if result["success"])
num_trials_without_collision = sum(
    1 for result in trial_results if not result["has_collided"]
)
num_successful_trials_without_collision = sum(
    1 for result in trial_results if result["success"] and not result["has_collided"]
)
num_code_errors = sum(
    1 for result in trial_results if result["trial_duration"] == float("inf")
)
num_code_errors_with_collision = sum(
    1
    for result in trial_results
    if result["trial_duration"] == float("inf") and result["has_collided"]
)

plt.figure()
plt.title("Pick-and-Place Policy Evaluation Results")
plt.bar(
    [
        "Successful Trials",
        "Trials without Collision",
        "Successful Trials without Collision",
        "Trials with Code Errors",
        "Trials with Code Errors and Collision",
    ],
    [
        num_successful_trials / NUM_TRIALS * 100,
        num_trials_without_collision / NUM_TRIALS * 100,
        num_successful_trials_without_collision / NUM_TRIALS * 100,
        num_code_errors / NUM_TRIALS * 100,
        num_code_errors_with_collision / NUM_TRIALS * 100,
    ],
)

#rotate x labels 45 degrees for better readability
plt.xticks(rotation=90)
plt.ylim(0, 100)
plt.ylabel("% of trials")
plt.yticks(range(0, 101, 5))
plt.show()

Below we visualize average trial durations.

**NOTE**: you are expected to complete trials in less than 45 seconds on average (planning time + simulation running time), and the simulation time should be less than 30 seconds on average (real-world execution time without planning).

In [ ]:
trials_without_error = [result for result in trial_results if result["trial_duration"] != float("inf")]

avg_trial_duration = np.mean(
    [result["trial_duration"] for result in trials_without_error]
) if trials_without_error else float("inf")
std_trial_duration = np.std(
    [result["trial_duration"] for result in trials_without_error]
) if trials_without_error else float("inf")

avg_trial_duration_no_collision = np.mean(
    [result["trial_duration"] for result in trials_without_error if not result["has_collided"]]
) if trials_without_error else float("inf")
std_trial_duration_no_collision = np.std(
    [result["trial_duration"] for result in trials_without_error if not result["has_collided"]]
) if trials_without_error else float("inf")

avg_trial_duration_with_collision = np.mean(
    [result["trial_duration"] for result in trials_without_error if result["has_collided"]]
) if trials_without_error else float("inf")
std_trial_duration_with_collision = np.std(
    [result["trial_duration"] for result in trials_without_error if result["has_collided"]]
) if trials_without_error else float("inf")

avg_trial_duration_successful = np.mean(
    [result["trial_duration"] for result in trials_without_error if result["success"]]
) if trials_without_error else float("inf")
std_trial_duration_successful = np.std(
    [result["trial_duration"] for result in trials_without_error if result["success"]]
) if trials_without_error else float("inf")

avg_trial_duration_failed = np.mean(
    [result["trial_duration"] for result in trials_without_error if not result["success"]]
) if trials_without_error else float("inf")
std_trial_duration_failed = np.std(
    [result["trial_duration"] for result in trials_without_error if not result["success"]]
) if trials_without_error else float("inf")

avg_trial_duration_successful_no_collision = np.mean(
    [result["trial_duration"] for result in trials_without_error if result["success"] and not result["has_collided"]]
) if trials_without_error else float("inf")
std_trial_duration_successful_no_collision = np.std(
    [result["trial_duration"] for result in trials_without_error if result["success"] and not result["has_collided"]]
) if trials_without_error else float("inf")

avg_simulation_time = np.mean(
    [result["simulation_time"] for result in trials_without_error]
) if trials_without_error else float("inf")
std_simulation_time = np.std(
    [result["simulation_time"] for result in trials_without_error]
) if trials_without_error else float("inf")

avg_simulation_time_no_collision = np.mean(
    [result["simulation_time"] for result in trials_without_error if not result["has_collided"]]
) if trials_without_error else float("inf")
std_simulation_time_no_collision = np.std(
    [result["simulation_time"] for result in trials_without_error if not result["has_collided"]]
) if trials_without_error else float("inf")

avg_simulation_time_with_collision = np.mean(
    [result["simulation_time"] for result in trials_without_error if result["has_collided"]]
) if trials_without_error else float("inf")
std_simulation_time_with_collision = np.std(
    [result["simulation_time"] for result in trials_without_error if result["has_collided"]]
) if trials_without_error else float("inf")

avg_simulation_time_successful = np.mean(
    [result["simulation_time"] for result in trials_without_error if result["success"]]
) if trials_without_error else float("inf")
std_simulation_time_successful = np.std(
    [result["simulation_time"] for result in trials_without_error if result["success"]]
) if trials_without_error else float("inf")

avg_simulation_time_failed = np.mean(
    [result["simulation_time"] for result in trials_without_error if not result["success"]]
) if trials_without_error else float("inf")
std_simulation_time_failed = np.std(
    [result["simulation_time"] for result in trials_without_error if not result["success"]]
) if trials_without_error else float("inf")

avg_simulation_time_successful_no_collision = np.mean(
    [result["simulation_time"] for result in trials_without_error if result["success"] and not result["has_collided"]]
) if trials_without_error else float("inf")
std_simulation_time_successful_no_collision = np.std(
    [result["simulation_time"] for result in trials_without_error if result["success"] and not result["has_collided"]]
) if trials_without_error else float("inf")

avg_execution_time = np.mean(
    [result["trial_duration"] + result["simulation_time"] for result in trials_without_error]
) if trials_without_error else float("inf")
std_execution_time = np.std(
    [result["trial_duration"] + result["simulation_time"] for result in trials_without_error]
) if trials_without_error else float("inf")

plt.figure()
plt.title("Average Trial Durations")
plt.bar(
    [
        "All Trials",
        "Trials without Collision",
        "Successful Trials",
        "Successful Trials without Collision",
        "Failed Trials",
        "Trials with Collision",
    ],
    [
        avg_trial_duration,
        avg_trial_duration_no_collision,
        avg_trial_duration_successful,
        avg_trial_duration_successful_no_collision,
        avg_trial_duration_failed,
        avg_trial_duration_with_collision,
    ],
    yerr=[
        std_trial_duration,
        std_trial_duration_no_collision,
        std_trial_duration_successful,
        std_trial_duration_successful_no_collision,
        std_trial_duration_failed,
        std_trial_duration_with_collision,
    ],
)
plt.ylabel("Duration (seconds)")
plt.xticks(rotation=90)
plt.show()

plt.figure()
plt.title("Average Simulation Times")
plt.bar(
    [
        "All Trials",
        "Trials without Collision",
        "Successful Trials",
        "Successful Trials without Collision",
        "Failed Trials",
        "Trials with Collision",
    ],
    [
        avg_simulation_time,
        avg_simulation_time_no_collision,
        avg_simulation_time_successful,
        avg_simulation_time_successful_no_collision,
        avg_simulation_time_failed,
        avg_simulation_time_with_collision,
    ],
    yerr=[
        std_simulation_time,
        std_simulation_time_no_collision,
        std_simulation_time_successful,
        std_simulation_time_successful_no_collision,
        std_simulation_time_failed,
        std_simulation_time_with_collision,
    ],
)
plt.ylabel("Duration (seconds)")
plt.xticks(rotation=90)
plt.show()

## Conclusion

In this assignment, we implemented a full pick-and-place task using motion planning and PID control in configuration space. We covered inverse kinematics, PID control, motion planning in changing environments and gripper control, and task execution using a sequential state machine. We combined all these components into a single policy that successfully executed the pick-and-place task, which we rendered smoothly in simulation. Just remember that the smooth result you see in the video does not reflect the true execution because the time spent planning is not shown. In the real world, planning occurs in real-time, and the robot must wait for the planner to find a solution before executing the motion.

The interesting thing about the pipeline implemented in this assignment is their generality. The block can be anywhere on the table, the bin can be anywhere in the robot's reachable workspace, and the obstacles can be arranged in any configuration. While the empirical success rate may seem underwhelming, with further tuning and improvements to each component, this pipeline can be made robust enough for practical use in real-world scenarios.

And unlike learning-based methods, this pipeline:
1. Does not require any training data.
2. Can be adapted to new robots and environments with minimal effort by changing the input model.
3. Each component of the pipeline can be improved independently.
4. Is interpretable and explainable, making it easier to diagnose and fix issues.

What's missing from this pipeline? Where did we cheat the most? Notice that we are highly dependent on data from the simulator. This includes the geometry and pose of the obstacles, bin, box, and table. We use this to choose configurations for grasping and to perform collision checking during motion planning. In the real world, this information is not readily available and must be estimated using sensors such as cameras and LIDAR. This introduces uncertainty and noise into the system, which can significantly affect the performance of the pipeline. Addressing this challenge requires integrating perception and uncertainty handling into the pipeline. We will cover these topics later in the course.